# E3 → E4 — Nube de puntos rota → STL imprimible

Pipeline completo de inferencia y reconstrucción de malla:

```
[.npy roto (2048,3)]  →  PoinTr  →  [nube completa (2048,3)]  →  Open3D Poisson  →  [malla .STL watertight]
```

**Celda 3** — configuración: elige modelo y muestra de entrada  
**Celda 6** — inferencia + visualización 3D de la nube  
**Celda 7** — reconstrucción de malla (Poisson)  
**Celda 8** — limpieza + exportar STL  
**Celda 9** — visualizar malla 3D interactiva

In [31]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

Drive ya montado.


In [32]:
# ── CELDA 2: Clonar PoinTr + repo TFM + instalar dependencias ──
import os, subprocess
from getpass import getpass

if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')
else:
    print('[OK] PoinTr ya existe.')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR,'-q'], capture_output=True)
    del token; print('Repo TFM clonado.')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)
    print('[OK] TFM actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)

# Dependencias
subprocess.run(['pip','install','timm','easydict','pyyaml','plotly','--quiet'])
r = subprocess.run(['pip','install','open3d','--quiet'], capture_output=True, text=True)
print(f'[{"OK" if r.returncode==0 else "WARN"}] open3d')
r = subprocess.run(['pip','install','trimesh','--quiet'], capture_output=True, text=True)
print(f'[{"OK" if r.returncode==0 else "WARN"}] trimesh')

for ext_name, ext_path in [
    ('pointnet2_ops', '/content/PoinTr/extensions/pointnet2_ops_lib'),
    ('chamfer_dist',  '/content/PoinTr/extensions/chamfer_dist'),
]:
    r = subprocess.run(['pip','install','-e',ext_path,'--quiet'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN fallback"}] {ext_name}')

[OK] PoinTr ya existe.
[OK] TFM actualizado.
[WARN] open3d
[OK] trimesh
[WARN fallback] pointnet2_ops
[OK] chamfer_dist


In [33]:
# ══════════════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACION
# ══════════════════════════════════════════════════════════════

# Modelo a usar (el mejor que tenemos)
VERSION = 'v5_obj'   # cambiar a 'v6_obj_sn' cuando esté listo

# ── Muestra de entrada ─────────────────────────────────────────
# Opción A: None → escoge aleatoriamente del test set del modelo
# Opción B: ruta a un .npy concreto (subido a Drive o en el repo)
RUTA_ROTO = None
# RUTA_ROTO = '/content/drive/MyDrive/Datos_E2_E3/General/Fantastik_Break_Procesado_v2/mug_01_roto.npy'

N_MUESTRAS = 4   # cuántas muestras mostrar si RUTA_ROTO=None

# ── Parámetros de reconstrucción de malla ──────────────────────
POISSON_DEPTH    = 9     # 8=rápido/suave | 9=detalle normal | 10=mucho detalle
DENSIDAD_CORTE   = 10    # percentil inferior a eliminar (0–30; más alto = malla más limpia)
RELLENAR_HUECOS  = True  # intentar cerrar huecos pequeños para watertight

# ── Rutas de checkpoints ───────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GEN= f'{DRIVE}/Datos_E2_E3/General'

RUTA_MODELO_DRIVE = f'{BASE_E3}/modelos/{VERSION}/best.pt'
RUTA_MODELO_LOCAL = f'E3/checkpoints_pointr_{VERSION}/best.pt'

SALIDA_DIR = f'E3/stl_generados/{VERSION}'

# ── Datasets del modelo elegido (para reconstruir el test set) ──
_VERSION_DATASETS = {
    'v5_obj':    ['obj'],
    'v5_fb_obj': ['fb', 'obj'],
    'v5_all':    ['fb', 'obj', 'sn'],
    'v6_obj_sn': ['obj', 'sn'],
    'v6_all':    ['fb', 'obj', 'sn'],
}
_partes = _VERSION_DATASETS.get(VERSION, ['obj'])

print(f'Modelo  : PoinTr {VERSION}')
print(f'Datasets: {_partes}')
print(f'Entrada : {RUTA_ROTO or f"aleatoria del test set ({N_MUESTRAS} muestras)"}')
print(f'Poisson depth={POISSON_DEPTH}  densidad_corte={DENSIDAD_CORTE}%')

Modelo  : PoinTr v5_obj
Datasets: ['obj']
Entrada : aleatoria del test set (4 muestras)
Poisson depth=9  densidad_corte=10%


In [34]:
# ── CELDA 4: Mocks CUDA + cargar modelo ────────────────────────
import sys, os, types, glob as _glob
from pathlib import Path

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

import torch
import torch.nn as nn
from easydict import EasyDict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')

# Reset módulos PoinTr cacheados
_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]

# Patch .cuda()
_np = 0
for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src = open(fp, encoding='utf-8').read()
        new = src.replace('.cuda()', f'.to("{DEVICE_STR}")')
        if new != src: open(fp,'w',encoding='utf-8').write(new); _np+=1
    except: pass

# Mocks CUDA
def _force(name, attrs):
    m = types.ModuleType(name)
    for k,v in attrs.items(): setattr(m,k,v)
    sys.modules[name] = m

def _inject(name, attrs):
    if name not in sys.modules: _force(name, attrs)

def _chamfer_raw(a,b):
    d=torch.cdist(a,b,p=2); return d.min(2).values, d.min(1).values

class _ChamferL1(nn.Module):
    def forward(self,a,b):
        d1,d2=_chamfer_raw(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
class _ChamferL2(nn.Module):
    def forward(self,a,b):
        d1,d2=_chamfer_raw(a.contiguous(),b.contiguous()); return ((d1**2).mean()+(d2**2).mean())/2
class _ChamferL1_PM(nn.Module):
    def forward(self,a,b): return torch.cdist(a.contiguous(),b.contiguous(),p=2).min(2).values.mean()

_ch={'ChamferDistanceL1':_ChamferL1,'ChamferDistanceL2':_ChamferL2,
     'ChamferDistanceL1_PM':_ChamferL1_PM,'chamfer_3DDist':_chamfer_raw}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']:
    _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np_):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np_,dtype=torch.int32,device=dev)
        dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev)
        bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np_):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
            dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})

def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']: _inject(pfx+base,attrs)

print('[OK] Mocks CUDA listos')

# ── Cargar modelo ─────────────────────────────────────────────
from pathlib import Path

ckpt_path = RUTA_MODELO_LOCAL
if not Path(ckpt_path).exists():
    ckpt_path = RUTA_MODELO_DRIVE
    print(f'Cargando desde Drive: {ckpt_path}')
else:
    print(f'Cargando desde local: {ckpt_path}')

ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Checkpoint epoca {ck["epoch"]}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device).eval()
print(f'[OK] Modelo PoinTr {VERSION} cargado — {sum(p.numel() for p in model.parameters()):,} parámetros')

Dispositivo: cuda
[OK] Mocks CUDA listos
Cargando desde Drive: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/modelos/v5_obj/best.pt


2026-08-27 20:11:35,924 - MODEL - INFO -  Transformer with knn_layer 1
2026-08-27 20:11:35,924 - MODEL - INFO -  Transformer with knn_layer 1
2026-08-27 20:11:35,924 - MODEL - INFO -  Transformer with knn_layer 1
2026-08-27 20:11:35,924 - MODEL - INFO -  Transformer with knn_layer 1
2026-08-27 20:11:35,924 - MODEL - INFO -  Transformer with knn_layer 1


Checkpoint epoca 257
[OK] Modelo PoinTr v5_obj cargado — 42,202,854 parámetros


In [35]:
# ── CELDA 5: Preparar muestra(s) de entrada ────────────────────
import numpy as np, random
from pathlib import Path
from E3.dataset import construir_pares

FUENTES_TODAS = {
    'fb':  (f'{BASE_GEN}/Fantastik_Break_Procesado_v2',
            'Datos/fantastic_breaks/procesado_v2'),
    'obj': (f'{BASE_GEN}/roturas_Objaverse_v2',
            'Datos/objaverse/roturas_v2'),
    'sn':  (f'{BASE_GEN}/shapenet_roturas',
            'Datos/shapenet/roturas'),
}
FUENTES_ACTIVAS = {k: v for k, v in FUENTES_TODAS.items() if k in _partes}

if RUTA_ROTO is not None:
    # Modo manual: una sola muestra
    ruta_r = RUTA_ROTO
    # Buscar el completo correspondiente (mismo nombre sin _roto)
    ruta_c = ruta_r.replace('_roto.npy', '_completo.npy')
    if not Path(ruta_c).exists():
        ruta_c = None
        print('⚠️  No se encontró el .npy completo — se mostrará solo la predicción.')
    muestras = [(ruta_r, ruta_c)]
    print(f'Muestra manual: {Path(ruta_r).name}')
else:
    # Modo automático: test set del modelo
    carpetas_locales = [v[1] for v in FUENTES_ACTIVAS.values() if Path(v[1]).exists()]
    if not carpetas_locales:
        # Intentar desde Drive directamente
        carpetas_locales = [v[0] for v in FUENTES_ACTIVAS.values() if Path(v[0]).exists()]
    if not carpetas_locales:
        raise RuntimeError('No hay datos — ejecuta la Celda 5 del notebook de entrenamiento primero, '
                           'o pon RUTA_ROTO con una ruta directa.')

    _todos = construir_pares(carpetas_locales)
    _rng = random.Random(42); _rng.shuffle(_todos)
    _n = len(_todos)
    _nt = int(0.8*_n); _nv = int(0.1*_n)
    _te = _todos[_nt+_nv:]

    # Escoger N_MUESTRAS aleatorias del test set
    muestras = random.sample(_te, min(N_MUESTRAS, len(_te)))
    print(f'Test set: {len(_te)} pares  |  Mostrando {len(muestras)} aleatorias')

print(f'\nMuestras a procesar:')
for i, (r,c) in enumerate(muestras):
    print(f'  [{i+1}] {Path(r).name}')

Test set: 41 pares  |  Mostrando 4 aleatorias

Muestras a procesar:
  [1] objaverse_29119fd73cf44d70a4f038849a25dfb5_r0_roto.npy
  [2] objaverse_f173eedc45904558824415d09646753e_r2_roto.npy
  [3] objaverse_ce6383d8fb27451eb8c39a7193194307_roto.npy
  [4] objaverse_9d5df9df7a24430888f683c9b2f74f81_r0_roto.npy


In [36]:
# ── CELDA 6: Inferencia + visualización 3D de la nube ──────────
import plotly.graph_objects as go
import numpy as np, torch
from pathlib import Path

def inferir(roto_np):
    with torch.no_grad():
        inp = torch.tensor(roto_np, dtype=torch.float32).unsqueeze(0).to(device)
        out = model(inp)
        return (out[-1] if isinstance(out,(list,tuple)) else out).squeeze(0).cpu().numpy()

def cd_metrica(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    d  = torch.cdist(pt, gt, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

def nube(pts, color, name, size=2, opacity=0.8):
    return go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers', name=name,
        marker=dict(size=size, color=color, opacity=opacity))

_escena = dict(
    xaxis=dict(range=[-1.2,1.2], showticklabels=False, title='',
               backgroundcolor='#111', gridcolor='#333', zerolinecolor='#333'),
    yaxis=dict(range=[-1.2,1.2], showticklabels=False, title='',
               backgroundcolor='#111', gridcolor='#333', zerolinecolor='#333'),
    zaxis=dict(range=[-1.2,1.2], showticklabels=False, title='',
               backgroundcolor='#111', gridcolor='#333', zerolinecolor='#333'),
    bgcolor='#111', aspectmode='cube'
)

resultados = []   # guardamos (nombre, pred_np, cd) para las celdas siguientes

for i, (ruta_r, ruta_c) in enumerate(muestras):
    roto = np.load(ruta_r).astype(np.float32)
    pred = inferir(roto)
    nombre = Path(ruta_r).stem.replace('_roto', '')

    cd = None
    titulo_extra = ''
    if ruta_c and Path(ruta_c).exists():
        comp = np.load(ruta_c).astype(np.float32)
        cd   = cd_metrica(pred, comp)
        titulo_extra = f'  CD={cd:.4f}'

    resultados.append({'nombre': nombre, 'pred': pred, 'roto': roto,
                       'comp': comp if (ruta_c and Path(ruta_c).exists()) else None,
                       'cd': cd})

    fig = go.Figure()
    if ruta_c and Path(ruta_c).exists():
        fig.add_trace(nube(comp, '#42A5F5', 'GT completo',    size=1.5, opacity=0.12))
    fig.add_trace(nube(pred, '#66BB6A', 'Predicción PoinTr', size=2,   opacity=0.85))
    fig.add_trace(nube(roto, '#EF5350', 'Roto (entrada)',    size=3,   opacity=0.95))

    fig.update_layout(
        scene=_escena,
        title=dict(
            text=f'<b>[{i+1}/{len(muestras)}] {nombre}</b>{titulo_extra}  | PoinTr {VERSION}',
            font=dict(color='white', size=12), x=0.5),
        legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)', x=0.01, y=0.99),
        paper_bgcolor='#111', height=600, width=680,
        margin=dict(l=0,r=0,t=50,b=0)
    )
    fig.show()
    print(f'  [{i+1}] {nombre}  —  {len(pred)} puntos predichos{titulo_extra}')

print(f'\n{len(resultados)} nubes listas. Pasa a Celda 7 para generar la malla.')

  [1] objaverse_29119fd73cf44d70a4f038849a25dfb5_r0  —  4096 puntos predichos  CD=0.0482


  [2] objaverse_f173eedc45904558824415d09646753e_r2  —  4096 puntos predichos  CD=0.0234


  [3] objaverse_ce6383d8fb27451eb8c39a7193194307  —  4096 puntos predichos  CD=0.0263


  [4] objaverse_9d5df9df7a24430888f683c9b2f74f81_r0  —  4096 puntos predichos  CD=0.0264

4 nubes listas. Pasa a Celda 7 para generar la malla.


In [37]:
# ── CELDA 7: Nube de puntos → Malla 3D (Poisson con pymeshlab) ─
import subprocess
subprocess.run(['pip', 'install', 'pymeshlab', '-q'])

import pymeshlab
import numpy as np
from pathlib import Path

Path(SALIDA_DIR).mkdir(parents=True, exist_ok=True)
mallas = []

for r in resultados:
    nombre = r['nombre']
    pred = r['pred']
    print("\n--", nombre)

    ms = pymeshlab.MeshSet()
    m = pymeshlab.Mesh(vertex_matrix=pred.astype(np.float64))
    ms.add_mesh(m, nombre)

    ms.compute_normal_for_point_clouds(k=30, smoothiter=2)
    print("  Normales calculadas")

    ms.generate_surface_reconstruction_screened_poisson(depth=POISSON_DEPTH, scale=1.1)
    n_v = ms.current_mesh().vertex_number()
    n_f = ms.current_mesh().face_number()
    print("  Poisson OK:", n_v, "vertices,", n_f, "caras")

    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=500)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)

    # Guardar STL
    ruta_stl = str(Path(SALIDA_DIR) / (nombre + '.stl'))
    ms.save_current_mesh(ruta_stl)
    tam_kb = Path(ruta_stl).stat().st_size / 1024
    print("  STL guardado:", Path(ruta_stl).name, f"({tam_kb:.0f} KB)")

    # Comprobar watertight con trimesh (pymeshlab no tiene is_water_tight)
    wt = False
    try:
        import trimesh
        tm = trimesh.load(ruta_stl, force='mesh')
        wt = bool(tm.is_watertight)
    except Exception:
        pass
    print("  Watertight:", "SI" if wt else "NO")

    verts = ms.current_mesh().vertex_matrix()
    faces = ms.current_mesh().face_matrix()
    mallas.append({'nombre': nombre, 'verts': verts, 'faces': faces,
                   'watertight': wt, 'cd': r['cd'], 'stl': ruta_stl,
                   'tam_kb': round(tam_kb, 1)})

print("\n" + str(len(mallas)) + " mallas generadas. Pasa a Celda 8 para copiar a Drive.")


-- objaverse_29119fd73cf44d70a4f038849a25dfb5_r0
  Normales calculadas
  Poisson OK: 11737 vertices, 23269 caras
  STL guardado: objaverse_29119fd73cf44d70a4f038849a25dfb5_r0.stl (1122 KB)
  Watertight: NO

-- objaverse_f173eedc45904558824415d09646753e_r2
  Normales calculadas
  Poisson OK: 24616 vertices, 48976 caras
  STL guardado: objaverse_f173eedc45904558824415d09646753e_r2.stl (2314 KB)
  Watertight: NO

-- objaverse_ce6383d8fb27451eb8c39a7193194307
  Normales calculadas
  Poisson OK: 21733 vertices, 43185 caras
  STL guardado: objaverse_ce6383d8fb27451eb8c39a7193194307.stl (2055 KB)
  Watertight: NO

-- objaverse_9d5df9df7a24430888f683c9b2f74f81_r0
  Normales calculadas
  Poisson OK: 13329 vertices, 26468 caras
  STL guardado: objaverse_9d5df9df7a24430888f683c9b2f74f81_r0.stl (1292 KB)
  Watertight: NO

4 mallas generadas. Pasa a Celda 8 para copiar a Drive.


In [38]:
# ── CELDA 8: Copiar STLs a Drive + resumen ────────────────────
import json, shutil
from pathlib import Path
from datetime import date

# Copiar a Drive
drive_stl = Path(f'{BASE_E3}/stl/{VERSION}')
try:
    drive_stl.mkdir(parents=True, exist_ok=True)
    for stl in Path(SALIDA_DIR).glob('*.stl'):
        shutil.copy2(stl, drive_stl / stl.name)
    print("STLs copiados a Drive:", str(drive_stl))
except Exception as e:
    print("[WARN] No se pudo copiar a Drive:", e)

# Resumen JSON
resumen = {
    'modelo': VERSION,
    'fecha': str(date.today()),
    'poisson_depth': POISSON_DEPTH,
    'mallas': [
        {'nombre': r['nombre'], 'stl': r['stl'], 'watertight': r['watertight'],
         'vertices': len(r['verts']), 'triangulos': len(r['faces']),
         'tam_kb': r['tam_kb'], 'cd_l1': r['cd']}
        for r in mallas
    ]
}
ruta_resumen = Path(SALIDA_DIR) / 'resumen.json'
with open(ruta_resumen, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2)

print("\n=== RESUMEN ===")
print(f"{'Nombre':<35} {'Verts':>7} {'Tris':>7} {'KB':>6} {'Water':>6} {'CD':>8}")
print('-' * 70)
for r in resumen['mallas']:
    cd_str = f"{r['cd_l1']:.4f}" if r['cd_l1'] else '—'
    wt_str = 'SI' if r['watertight'] else 'NO'
    print(f"{r['nombre'][:34]:<35} {r['vertices']:>7} {r['triangulos']:>7} "
          f"{r['tam_kb']:>6.0f} {wt_str:>6} {cd_str:>8}")

STLs copiados a Drive: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/stl/v5_obj

=== RESUMEN ===
Nombre                                Verts    Tris     KB  Water       CD
----------------------------------------------------------------------
objaverse_29119fd73cf44d70a4f03884    11581   22985   1122     NO   0.0482
objaverse_f173eedc45904558824415d0    23775   47382   2314     NO   0.0234
objaverse_ce6383d8fb27451eb8c39a71    21153   42079   2055     NO   0.0263
objaverse_9d5df9df7a24430888f683c9    13317   26452   1292     NO   0.0264


In [39]:
# ── CELDA 9: Visualizar malla 3D interactiva ──────────────────
import plotly.graph_objects as go
import numpy as np

for r in mallas:
    verts = r['verts']
    faces = r['faces']
    nombre = r['nombre']

    if len(verts) == 0 or len(faces) == 0:
        print("Malla vacia:", nombre)
        continue

    fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        colorscale=[[0,'#1a6b8a'],[0.5,'#4ecdc4'],[1,'#e8f4f8']],
        intensity=verts[:,2],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.8, roughness=0.5, specular=0.4),
        lightposition=dict(x=100, y=200, z=300)
    ))

    cd_str = f"CD={r['cd']:.4f}  " if r['cd'] else ''
    wt_str = 'watertight' if r['watertight'] else 'con huecos'
    fig.update_layout(
        title=f"{nombre}  {cd_str}{wt_str}  |  {len(verts)} verts / {len(faces)} caras",
        scene=dict(aspectmode='data', bgcolor='#1a1a2e',
                   xaxis=dict(showticklabels=False, title=''),
                   yaxis=dict(showticklabels=False, title=''),
                   zaxis=dict(showticklabels=False, title='')),
        paper_bgcolor='#1a1a2e',
        font=dict(color='white'),
        height=620, width=700,
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()
    print(" ", nombre, "|", len(verts), "verts |", len(faces), "caras |", wt_str)

print("\nSTLs en:", SALIDA_DIR)
from pathlib import Path
for stl in Path(SALIDA_DIR).glob('*.stl'):
    print(" ", stl.name, f"({stl.stat().st_size/1024:.0f} KB)")

  objaverse_29119fd73cf44d70a4f038849a25dfb5_r0 | 11581 verts | 22985 caras | con huecos


  objaverse_f173eedc45904558824415d09646753e_r2 | 23775 verts | 47382 caras | con huecos


  objaverse_ce6383d8fb27451eb8c39a7193194307 | 21153 verts | 42079 caras | con huecos


  objaverse_9d5df9df7a24430888f683c9b2f74f81_r0 | 13317 verts | 26452 caras | con huecos

STLs en: E3/stl_generados/v5_obj
  objaverse_f173eedc45904558824415d09646753e_r2.stl (2314 KB)
  objaverse_29119fd73cf44d70a4f038849a25dfb5_r0.stl (1122 KB)
  objaverse_ce6383d8fb27451eb8c39a7193194307.stl (2055 KB)
  objaverse_9d5df9df7a24430888f683c9b2f74f81_r0.stl (1292 KB)
